In [ ]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/raw/2026/bikeshare-ridership-2026.csv")

In [2]:
df.describe()

,Trip_Id,Trip_Duration,Start_Station_Id,End_Station_Id,Bike_Id
count,6.830174e+06,6.830174e+06,6.830174e+06,6.817223e+06,6.830174e+06
mean,4.759683e+07,1.010622e+03,7.465718e+03,7.461785e+03,6.512163e+03
std,2.265120e+06,1.607860e+03,3.635928e+02,3.625551e+02,3.472318e+03
min,4.368242e+07,0.000000e+00,7.000000e+03,7.000000e+03,1.400000e+01
25%,4.563085e+07,4.330000e+02,7.149000e+03,7.147000e+03,3.511000e+03
50%,4.759595e+07,7.050000e+02,7.384000e+03,7.378000e+03,6.897000e+03
75%,4.955941e+07,1.149000e+03,7.768000e+03,7.762000e+03,9.571000e+03
max,5.151359e+07,2.592863e+06,8.273000e+03,8.273000e+03,1.197300e+04


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6830174 entries, 0 to 6830173
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    int64  
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(1), int64(4), str(6)
memory usage: 1.2 GB


In [4]:
df.columns

Index(['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time',
       'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name',
       'Bike_Id', 'User_Type', 'Bike_Model'],
      dtype='str')

In [5]:
df.index

RangeIndex(start=0, stop=6830174, step=1)

In [6]:
df[["User_Type", "Bike_Model"]]

,User_Type,Bike_Model
0,Member,ICONIC
1,Member,ICONIC
2,Member,ICONIC
3,Casual,ICONIC
4,Member,ICONIC
...,...,...
6830169,Casual,ICONIC
6830170,Member,EFIT
6830171,Member,ICONIC
6830172,Member,ICONIC


In [7]:
set(df["Bike_Model"])

{'ASTRO', 'EFIT', 'ICONIC'}

In [8]:
df.iloc[0]

Trip_Id                                43682415
Trip_Duration                               248
Start_Station_Id                           7719
Start_Time                  2026-01-01 00:00:26
Start_Station_Name    Wolseley St / Augusta Ave
End_Station_Id                           7037.0
End_Time                    2026-01-01 00:04:34
End_Station_Name      Bathurst St / Dundas St W
Bike_Id                                     122
User_Type                                Member
Bike_Model                               ICONIC
Name: 0, dtype: object

In [9]:
df.isna().sum()

Trip_Id                   0
Trip_Duration             0
Start_Station_Id          0
Start_Time                0
Start_Station_Name        0
End_Station_Id        12951
End_Time               8065
End_Station_Name      12951
Bike_Id                   0
User_Type                 0
Bike_Model                0
dtype: int64

In [10]:
df = df[df["Trip_Duration"] >= 60]

In [11]:
df["Trip_Duration"].max()

np.int64(2592863)

In [12]:
df["Trip_Duration"].quantile([0.99,0.999,0.9999])

0.9900     5794.000
0.9990    12875.000
0.9999    33666.241
Name: Trip_Duration, dtype: float64

In [13]:
(df["Trip_Duration"] >= 33666.241).sum()

np.int64(681)

In [14]:
(df["Trip_Duration"] >= 5794.00).sum()

np.int64(68091)

In [15]:
(df["Trip_Duration"] >= 12875.00).sum() 

np.int64(6809)

Took at look at the quantiles of the longest trips, and decided to remove trips over 3.6 hours (12875 seconds). Thought that there is a small chance that many are real and if not stolen or unlatched.

In [16]:
df = df[df["Trip_Duration"] <= 12875.00]

In [17]:
(df["Trip_Duration"] >= 12875.00).sum()

np.int64(4)

In [35]:
df.dtypes

Trip_Id                        int64
Trip_Duration                  int64
Start_Station_Id               int64
Start_Time            datetime64[us]
Start_Station_Name               str
End_Station_Id                 Int64
End_Time              datetime64[us]
End_Station_Name                 str
Bike_Id                        int64
User_Type                        str
Bike_Model                       str
is_ebike                        bool
dtype: object

In [19]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"])
df["End_Time"] = pd.to_datetime(df["End_Time"])

In [20]:
df["End_Station_Id"] = df["End_Station_Id"].astype("Int64")

Check for Duplicates, Converting DT to Date Time Type

In [21]:
(df["Trip_Id"].duplicated()).sum()

np.int64(0)

In [22]:
df["User_Type"].value_counts()

User_Type
Member    4652189
Casual    2148525
Name: count, dtype: int64

In [23]:
df["Bike_Model"].value_counts()

Bike_Model
ICONIC    5289972
EFIT      1442567
ASTRO       68175
Name: count, dtype: int64

Checking to see how many are round trips

In [24]:
(df["Start_Station_Id"] == df["End_Station_Id"]).sum()

np.int64(223054)

In [25]:
round_trips = df[df["Start_Station_Id"] == df["End_Station_Id"]]

In [26]:
round_trips["Trip_Duration"].describe()

count    223054.000000
mean       2511.683852
std        2325.406835
min          61.000000
25%         644.000000
50%        1881.000000
75%        3760.000000
max       12875.000000
Name: Trip_Duration, dtype: float64

In [27]:
(round_trips["Trip_Duration"] < 120).sum()

np.int64(16540)

Filtering Out Trips that are less than 120 seconds and redocks. Likely, there was an issue and it was returned.

In [28]:
is_redock =(df["Start_Station_Id"] == df["End_Station_Id"]) & (df["Trip_Duration"] < 120)
df = df[~is_redock]

In [31]:
df["is_ebike"] = df["Bike_Model"].isin(["EFIT","ASTRO"])

In [37]:
df.columns

Index(['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time',
       'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name',
       'Bike_Id', 'User_Type', 'Bike_Model', 'is_ebike'],
      dtype='str')

In [33]:
len(df)

6784133

In [1]:
conn = sqlite3.connect("../data/processed/bikeshare.db")
df.to_sql("trips", conn, if_exists="replace", index=False)
conn.close()

NameError: name 'sqlite3' is not defined